# E14b: retraining with the slice density of MCT-LTDiag

**Revision plan, 00 log section 60 (written 23 Oct 2026, before this run).** E12 retrained every model with the slice density
equalized across classes at the development median. E14b repeats E12 exactly, with **two changes**: the common target is the
slice density of the external cohort, **R = 0.11698** (E10b's estimate, slices containing liver ÷ largest in-plane liver
width; median of the 203 MCT-LTDiag patients, E12 file), and **scans are made thicker only by slab averaging** (each new slice
is the overlap-weighted mean of the original slices it covers, as a thick reconstruction; masks averaged the same way, then
> 0.5). Scans that are already thicker are interpolated linearly, as in E12 (MCT-LTDiag's factors are close to 1). Folds,
seeds and settings are E12's; nothing is tuned. **The main models and all main results stay unchanged, whatever this shows.**

- **Retrained:** phase-dropout CNN (E3 protocol), E2 radiomics (features re-extracted), CNN + radiomics and full fusion.
- **O0** harmonization (class medians of the ratio after the change; a slice-count-only model before and after). **O1**
  internal held-out and out-of-fold results, paired against the main models. **O2** residual artifact (E12's test). **O3**
  MCT-LTDiag (203), HCC versus ICC, paired DeLong test against the main models, the AUC against 0.5, and predicted classes.
- **Rules fixed in the plan:** harmonization succeeded if the class medians differ at most 1.25-fold in every phase; an
  internal change is claimed only if the held-out paired bootstrap CI excludes 0; an external improvement only if the paired
  DeLong p < 0.05 with a higher AUC. **Interpretation:** internal performance kept and MCT-LTDiag not improved → acquisition
  geometry is not the main cause of the external failure; MCT-LTDiag improved → acquisition mismatch contributed; internal
  performance fell → internal discrimination depended on class-dependent acquisition.
- **Gates (the notebook stops if one fails):** E12's gates, plus R equals the plan value and the recomputed MCT-LTDiag median
  ratio equals R.

**Runtime:** **A100 GPU with High-RAM**, about 3–4 hours. **Resume:** if the run stops, set `RESUME_RUN_ID` in Cell 1 to the
stopped run's time and run all again; trained folds and extracted features are reused. **When it finishes,** the last cell
saves everything to Drive and disconnects the runtime (`SHUTDOWN_WHEN_DONE = False` keeps it). **If a cell stops with an
error,** the error is saved to Drive (`E14b_error.txt`) and the runtime disconnects (`SHUTDOWN_ON_ERROR = False` in Cell 1 keeps
it). **Safety:** inputs are only read; outputs go to a new folder `liver_project/revision_2026-10/E14b_<time>/`.

In [ ]:
# Cell 1 — Setup (E6's cell): installs, paths, input checks, output folder E14b_<time>, disconnect on error
import os, sys, io, re, json, time, shutil, tarfile, hashlib, platform, datetime, warnings, subprocess
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    WORK = Path('/content/e14b_work')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm==4.6.0', 'pydicom', 'nibabel'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E6_TEST_ROOT'])
    WORK = Path(os.environ['E6_TEST_WORK'])
WORK.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import nibabel as nib
import pydicom
import joblib
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy import ndimage
from scipy.stats import beta, norm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
MCT_DIR = PROJECT_DIR / 'external_validation' / 'MCT-LTDiag'
E8B_DIR = REVISION_DIR / 'E8b_20261018T025705Z'
E10_DIR = REVISION_DIR / 'E10_20261019T031225Z' if IN_COLAB else Path(os.environ['E14B_TEST_E10'])
E12_DIR = REVISION_DIR / 'E12_20261022T025246Z' if IN_COLAB else Path(os.environ['E14B_TEST_E12'])
SEED = 42
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']
N_IO = 6 if IN_COLAB else 2                       # patients read/preprocessed at the same time
N_JOBS = os.cpu_count() if IN_COLAB else 1        # parallel radiomics processes

INPUTS = {
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_noshape_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_original_predictions': (E3_DIR / 'E3A_original_cnn_predictions.csv', '5c02c3ebda984b6283c1137d85c17c171347b24ae1fac450dfc89c5513d0229b'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'mct_selected': (MCT_DIR / 'metadata' / 'selected_HCC_ICC_patients.csv', 'fdb8d8fa74d8211bbfe45ecb4820415e52a5183ce5d51bcfa4376cfa9ed11e20'),
    'mct_patient_table': (MCT_DIR / 'metadata' / 'meta_info_patient.csv', 'f9f0fc730b1ec75cecb615304aa0f69db30225fb559b460ef78d0fe693220f0d'),
    'e2_test_meta': (E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', 'fbb946f641a050f66b093564141a8e103e1c4e5c651d6abeb51042293ab8cb2f'),
    'e2_radiomics_oof': (E2_DIR / 'E2_radiomics_noshape_oof_predictions.csv', '21057b42933293a7d45cbefbb6f58dfe89458064a42fff689abcbc826fe28cb6'),
    'e8b_predictions': (E8B_DIR / 'E8b_E9_predictions.csv', '644a520c0d034ce233bb88ab98e1b6d9e89993a661e380d06985edde4aa34064'),
    'e10b_anatomy': (E10_DIR / 'E10b_anatomy_per_scan.csv', 'f7ea15344beba14e25b85aa8fd6879e586670e6d311207c540d45f301ee4183c'),
    'e12_mct_geometry': (E12_DIR / 'E12_mct_crop_geometry.csv', '567bcc76a0eb6e89b6e1c866c793c857244beb1d915ded8afb814c7d96403f70'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
LARGE_INPUTS = {'processed_tar': (PROJECT_DIR / 'processed.tar', 402616320)}
MODEL_FILES = {
    **{f'w3_{f}': PROJECT_DIR / 'checkpoints_w3' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'w4_{f}': PROJECT_DIR / 'checkpoints_w4' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
    **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)},
}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

def md5_of(path):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
for key, (path, size) in LARGE_INPUTS.items():
    if not path.exists() or (IN_COLAB and path.stat().st_size != size):
        problems.append(f'missing or changed size: {path}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import sklearn, scipy, radiomics, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version_VersionString() if hasattr(sitk, 'Version_VersionString') else 'unknown',
            'pyradiomics': getattr(radiomics, '__version__', 'unknown'), 'pydicom': getattr(pydicom, '__version__', 'unknown'),
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': str(DEVICE), 'cpus': os.cpu_count()}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESUME_RUN_ID = None   # to continue a stopped run, put its time here, e.g. '20261020T020304Z'
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
OUT_DIR = REVISION_DIR / f'E14b_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=bool(RESUME_RUN_ID))

SHUTDOWN_ON_ERROR = True   # if any cell stops with an error: save it to Drive (E14b_error.txt) and disconnect the runtime
if IN_COLAB and SHUTDOWN_ON_ERROR:
    from IPython import get_ipython
    def _disconnect_on_error(shell, etype, evalue, tb, tb_offset=None):
        shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
        try:
            (OUT_DIR / 'E14b_error.txt').write_text(f'{etype.__name__}: {evalue}')
            from google.colab import drive as _drive, runtime as _runtime
            print('\nThe run stopped with the error above; it is saved in Drive (E14b_error.txt). The runtime disconnects '
                  'in 60 seconds to save compute units; this output stays visible. Send it for review.')
            _drive.flush_and_unmount()
            time.sleep(60)
            _runtime.unassign()
        except Exception as error:
            print('Could not disconnect automatically:', error)
    get_ipython().set_custom_exc((Exception,), _disconnect_on_error)
print('Inputs and models checked:', len(INPUTS) + len(LARGE_INPUTS) + len(MODEL_FILES))
print('Versions:', VERSIONS)
print('Output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Frozen models must reproduce their saved internal test predictions; internal orientation reference
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(LARGE_INPUTS['processed_tar'][0]), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E6_TEST_PROCESSED'])

split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
TEST_CT, TEST_LABEL = {}, {}
for pid in test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        TEST_CT[pid], TEST_LABEL[pid] = z['ct'].astype(np.uint8), int(z['label'])

def zscore_nonzero_per_phase(x4):          # Week 4, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def make_cnn(in_channels):
    return resnet18(spatial_dims=3, n_input_channels=in_channels, num_classes=3, feed_forward=True)

def load_models(keys, in_channels):
    models = []
    for key in keys:
        checkpoint = torch.load(MODEL_FILES[key], map_location=DEVICE, weights_only=False)
        model = make_cnn(in_channels).to(DEVICE)
        model.load_state_dict(checkpoint['state_dict'])
        model.eval()
        models.append(model)
    return models

CNNS = {'single_phase_CNN': (load_models([f'w3_{f}' for f in range(5)], 1), [2]),
        'original_CNN': (load_models([f'w4_{f}' for f in range(5)], 4), [0, 1, 2, 3]),
        'phase_dropout_CNN': (load_models([f'd_{f}' for f in range(5)], 4), [0, 1, 2, 3])}

@torch.no_grad()
def cnn_probs(models, channels, arrays):
    """Mean softmax over the 5 fold models; arrays: patient -> uint8 (4, 96, 96, 96) in internal axis order."""
    rows = {}
    for pid, ct in arrays.items():
        x = torch.from_numpy(zscore_nonzero_per_phase(ct)[channels]).unsqueeze(0).to(DEVICE)
        rows[pid] = np.mean([torch.softmax(m(x), dim=1).float().cpu().numpy()[0] for m in models], axis=0)
    return pd.DataFrame.from_dict(rows, orient='index', columns=['p_HCC', 'p_ICC', 'p_CHCC'])

def macro_auc3(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

y_test = [TEST_LABEL[p] for p in test_ids]
gate = {}
for name, key, model_tag in [('original_CNN', 'e3_original_predictions', 'original'),
                             ('phase_dropout_CNN', 'e3_d_predictions', 'D_phase_dropout')]:
    saved = pd.read_csv(INPUTS[key][0], dtype={'patient_id': str})
    saved = saved.query("pattern == 'complete' and split == 'test'").set_index('patient_id').loc[test_ids, ['p_HCC', 'p_ICC', 'p_CHCC']]
    ours = cnn_probs(*CNNS[name], {p: TEST_CT[p] for p in test_ids}).loc[test_ids]
    gate[name] = {'max_prob_diff': float(np.abs(ours.to_numpy() - saved.to_numpy()).max()),
                  'macro_auc_ours': macro_auc3(y_test, ours.to_numpy()), 'macro_auc_saved': macro_auc3(y_test, saved.to_numpy())}

internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
rad_models = [joblib.load(MODEL_FILES[f'rad_{f}']) for f in range(5)]
pool_cols = rad_models[0]['feature_pool']
X_int_test = internal.set_index('patient_id').loc[test_ids, pool_cols].to_numpy(dtype=np.float32)
p_int = np.mean([m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X_int_test[:, m['sel_idx']])))
                 for m in rad_models], axis=0)
saved_int = pd.read_csv(INPUTS['e2_noshape_test'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids, ['p_HCC', 'p_ICC', 'p_CHCC']].to_numpy()
gate['radiomics_noshape'] = {'max_prob_diff': float(np.abs(p_int - saved_int).max())}
GATE_OK = (all(gate[n]['max_prob_diff'] < 0.02 and abs(gate[n]['macro_auc_ours'] - gate[n]['macro_auc_saved']) < 0.005
               for n in ['original_CNN', 'phase_dropout_CNN']) and gate['radiomics_noshape']['max_prob_diff'] < 1e-6)
print('Internal reproduction check:', json.dumps(gate, indent=1))
if IN_COLAB and not GATE_OK:
    raise RuntimeError('STOP — the frozen models do not reproduce their saved internal test predictions.')

# ---- internal orientation reference (stored arrays, as Week 2 read them with nibabel) ----
def load_nii(path):                          # Week 2, unchanged
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

BONE_0255 = 254                               # >= ~198 HU on the -200..200 -> 0..255 scale

def orientation_descriptor(ct0255_ijk, liver_ijk):
    """Where the liver and the bone are in an (i, j, k) array, as fractions of each axis (no class labels)."""
    li, lj, lk = np.nonzero(liver_ijk)
    if li.size < 500:
        return None
    shape = liver_ijk.shape
    bone = (ct0255_ijk >= BONE_0255) & ~liver_ijk
    bi, bj, _ = np.nonzero(bone)
    area = liver_ijk.sum(axis=(0, 1)).astype(np.float64)
    k0, k1 = int(lk.min()), int(lk.max())
    centre_k = float((np.arange(len(area)) * area).sum() / area.sum())
    return {'liver_i': float(li.mean() / shape[0]), 'liver_j': float(lj.mean() / shape[1]),
            'bone_i': float(bi.mean() / shape[0]) if bi.size >= 200 else np.nan,
            'bone_j': float(bj.mean() / shape[1]) if bj.size >= 200 else np.nan,
            'bulk_k': (centre_k - k0) / max(1, k1 - k0)}

patients_table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
p_rows = patients_table[patients_table['phase'] == 'P'].drop_duplicates('patient_id')   # non-contrast: bone not confused with contrast

def internal_descriptor(row):
    try:
        return orientation_descriptor(load_nii(DATA_DIR / row['ct_path']), load_nii(DATA_DIR / row['liver_mask_path']) > 0)
    except Exception as error:
        return {'error': repr(error)[:120]}

with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    internal_desc = pd.DataFrame([d or {} for d in pool.map(internal_descriptor, [r for _, r in p_rows.iterrows()])])
internal_desc.insert(0, 'patient_id', p_rows['patient_id'].tolist())
internal_desc.to_csv(OUT_DIR / 'E14b_internal_orientation_reference.csv', index=False)
ok = internal_desc.dropna(subset=['liver_i', 'bone_i', 'bone_j'])
REF = {'front_back_axis_is_j': float((abs(ok['bone_j'] - .5) > abs(ok['bone_i'] - .5)).mean()),
       'liver_at_low_i': float((ok['liver_i'] < .5).mean()), 'bone_at_high_j': float((ok['bone_j'] > .5).mean()),
       'liver_bulk_at_low_k': float((ok['bulk_k'] < .5).mean()), 'n': int(len(ok)),
       'median_liver_i': float(ok['liver_i'].median()), 'median_bone_j': float(ok['bone_j'].median()),
       'median_bulk_k': float(ok['bulk_k'].median())}
print('Internal orientation reference:', json.dumps(REF, indent=1))
# Must agree with the 12-patient check in the 00 log (17.9): i = left-right with the liver at low i, spine at high j, bulk at low k
if not (REF['front_back_axis_is_j'] >= .8 and REF['liver_at_low_i'] >= .8 and REF['bone_at_high_j'] >= .8 and REF['liver_bulk_at_low_k'] >= .7):
    raise RuntimeError('STOP — the internal orientation reference differs from the 17 Oct check. Send this output.')

In [ ]:
# Cell 3 — E6's facts check for all 203 MCT-LTDiag scans, with the E8.0 grid fix
SEL = pd.read_csv(INPUTS['mct_selected'][0], dtype=str)
SEL['size'] = SEL['size'].astype(int)
counts = SEL['type'].value_counts().to_dict()
if IN_COLAB and counts != {'HCC': 103, 'ICC': 100}:
    raise RuntimeError(f'STOP — unexpected MCT-LTDiag selection {counts}.')
WANTED = {'NIFTI/nc.nii.gz': 'P', 'NIFTI/art.nii.gz': 'C1', 'NIFTI/pvp.nii.gz': 'C2', 'NIFTI/delay.nii.gz': 'C3',
          'mask_pvp.nii.gz': 'tumor', 'liver_mask_pvp.nii.gz': 'liver'}
DICOM_FIELDS = ['SeriesDescription', 'Manufacturer', 'ManufacturerModelName', 'SliceThickness', 'PixelSpacing', 'KVP',
                'ConvolutionKernel', 'SeriesInstanceUID']
TAR_TMP, NII_DIR = WORK / 'tar', WORK / 'nifti'
TAR_TMP.mkdir(exist_ok=True)
NII_DIR.mkdir(exist_ok=True)

def window_0255(hu):                          # PLC-CECT release convention: -200..200 HU -> 0..255
    return ((np.clip(hu, -200.0, 200.0) + 200.0) / 400.0 * 255.0).astype(np.float32)

def ijk(img):                                 # SimpleITK array (k, j, i) -> stored (i, j, k) order, as nibabel / Week 2
    arr = sitk.GetArrayFromImage(img)
    return np.ascontiguousarray(np.transpose(arr, (2, 1, 0)))

def grid(img):
    return img.GetSize(), img.GetSpacing(), img.GetOrigin(), img.GetDirection()

def same_grid(a, b):
    return (tuple(a[0]) == tuple(b[0]) and np.allclose(a[1], b[1], rtol=1e-3) and np.allclose(a[2], b[2], atol=0.5)
            and np.allclose(a[3], b[3], atol=1e-3))

def onto(img, reference, is_mask):
    return sitk.Resample(img, reference, sitk.Transform(), sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear,
                         0 if is_mask else -1024)

def nifti_facts(folder):
    imgs = {k: sitk.ReadImage(str(folder / f'{k}.nii.gz')) for k in PHASE_ORDER + ['tumor', 'liver']}
    ref = grid(imgs['C2'])
    f = {'size': list(ref[0]), 'spacing': [round(float(s), 4) for s in ref[1]], 'direction': [round(float(d), 3) for d in ref[3]],
         'off_grid': [k for k in imgs if k != 'C2' and not same_grid(grid(imgs[k]), ref)]}
    mm = sitk.MinimumMaximumImageFilter()
    for k in PHASE_ORDER:
        mm.Execute(imgs[k])
        sample = sitk.GetArrayViewFromImage(imgs[k])[::2, ::4, ::4]
        f[f'{k}_min'], f[f'{k}_max'] = float(mm.GetMinimum()), float(mm.GetMaximum())
        f[f'{k}_p50'] = float(np.percentile(sample, 50))
    mins, maxs = [f[f'{k}_min'] for k in PHASE_ORDER], [f[f'{k}_max'] for k in PHASE_ORDER]
    f['intensity_kind'] = ('HU' if max(mins) <= -500 else '0-255' if min(mins) >= 0 and max(maxs) <= 255 else 'unexpected')
    tumor_img = imgs['tumor'] if 'tumor' not in f['off_grid'] else onto(imgs['tumor'], imgs['C2'], is_mask=True)   # E8.0 fix
    liver_img0 = imgs['liver'] if 'liver' not in f['off_grid'] else onto(imgs['liver'], imgs['C2'], is_mask=True)
    tumor, liver = sitk.GetArrayFromImage(tumor_img), sitk.GetArrayFromImage(liver_img0)
    f['tumor_labels'] = [float(v) for v in np.unique(tumor)[:10]]
    f['liver_labels'] = [float(v) for v in np.unique(liver)[:10]]
    f['tumor_voxels'], f['liver_voxels'] = int((tumor > 0).sum()), int((liver > 0).sum())
    f['tumor_components'] = int(ndimage.label(tumor > 0)[1])
    f['tumor_volume_ml'] = f['tumor_voxels'] * float(np.prod(ref[1])) / 1000.0
    f['tumor_outside_liver_fraction'] = float(((tumor > 0) & (liver == 0)).sum() / max(1, (tumor > 0).sum()))
    plain = imgs['P'] if 'P' not in f['off_grid'] else onto(imgs['P'], imgs['C2'], is_mask=False)
    liver_img = imgs['liver'] if 'liver' not in f['off_grid'] else onto(imgs['liver'], imgs['C2'], is_mask=True)
    ct = ijk(plain)
    descriptor = orientation_descriptor(window_0255(ct) if f['intensity_kind'] == 'HU' else ct, ijk(liver_img) > 0)
    if descriptor is None or not np.isfinite(descriptor['bone_i']):
        raise RuntimeError('liver mask or bone too small to determine orientation')
    f.update(descriptor)
    f['nib_equals_sitk_ijk'] = bool(np.allclose(load_nii(folder / 'P.nii.gz'), ijk(imgs['P']).astype(np.float32), atol=1e-3))
    return f

def stage(row):
    pid = row['patient_id']
    folder = NII_DIR / pid
    done = folder / 'facts.json'
    if done.exists():
        return json.loads(done.read_text())
    folder.mkdir(exist_ok=True)
    facts, series = {'patient_id': pid, 'technical_failure': ''}, {}
    local = TAR_TMP / row['name']
    try:
        shutil.copyfile(MCT_DIR / f"tar_{row['type']}" / row['name'], local)
        if md5_of(local) != row['md5']:
            raise RuntimeError('checksum mismatch after copying from Drive')
        found = set()
        with tarfile.open(local, 'r:') as tf:
            for m in tf:
                if not m.isfile():
                    continue
                name = m.name[2:] if m.name.startswith('./') else m.name
                if name in WANTED:
                    (folder / f'{WANTED[name]}.nii.gz').write_bytes(tf.extractfile(m).read())   # fixed local names only
                    found.add(name)
                elif name.startswith('DICOM/'):
                    key = name.rsplit('/', 1)[0]
                    if key not in series:
                        try:
                            ds = pydicom.dcmread(io.BytesIO(tf.extractfile(m).read()), stop_before_pixels=True, force=True)
                            series[key] = {field: str(getattr(ds, field, '')) for field in DICOM_FIELDS}
                        except Exception as error:
                            series[key] = {'error': repr(error)[:100]}
                        series[key]['files'] = 0
                    series[key]['files'] += 1
        if set(WANTED) - found:
            raise RuntimeError(f'missing in tar: {sorted(set(WANTED) - found)}')
        facts.update(nifti_facts(folder))
        if facts['tumor_voxels'] == 0:
            facts['technical_failure'] = 'empty tumour mask'
    except Exception as error:
        facts['technical_failure'] = repr(error)[:200]
    finally:
        local.unlink(missing_ok=True)
    facts['dicom_series'] = list(series.values())
    done.write_text(json.dumps(facts, default=float))
    return facts

t0 = time.time()
rows = [r for _, r in SEL.iterrows()]
FACTS = []
with ThreadPoolExecutor(N_IO) as pool:
    for n, f in enumerate(pool.map(stage, rows), start=1):
        FACTS.append(f)
        if n % 20 == 0 or n == len(rows):
            print(f'  facts check {n}/{len(rows)} ({(time.time() - t0) / 60:.1f} min)', flush=True)

dicom_rows = [{'patient_id': f['patient_id'], **s} for f in FACTS for s in f.get('dicom_series', [])]
pd.DataFrame(dicom_rows).to_csv(OUT_DIR / 'E14b_mct_dicom_series.csv', index=False)
facts_df = pd.DataFrame([{k: v for k, v in f.items() if k != 'dicom_series'} for f in FACTS])
TECH_FAIL = facts_df.loc[facts_df['technical_failure'] != '', ['patient_id', 'technical_failure']]
INCLUDED = facts_df.loc[facts_df['technical_failure'] == '', 'patient_id'].tolist()
kinds = Counter(facts_df.loc[facts_df['technical_failure'] == '', 'intensity_kind'])
print(f'Included {len(INCLUDED)}/{len(SEL)}; technical failures: {TECH_FAIL.to_dict("records")}')
print('Intensity kind:', dict(kinds), '| phases off the portal-venous grid:',
      int(facts_df['off_grid'].map(lambda v: len(v) if isinstance(v, list) else 0).gt(0).sum()), 'patients')
if kinds.get('unexpected', 0):
    raise RuntimeError('STOP — some scans are neither HU nor 0-255 (rule 17.3-1). Send this output.')
if IN_COLAB and not facts_df.loc[facts_df['technical_failure'] == '', 'nib_equals_sitk_ijk'].all():
    raise RuntimeError('STOP — SimpleITK (i, j, k) arrays differ from nibabel arrays.')

# ---- orientation decisions (amendment 17.9-1): match each scan to the internal anatomy ----
def decide(d):
    # front-back axis = the in-plane axis on which the bone (spine) is off-centre; internally that is j
    off_i, off_j = abs(d['bone_i'] - .5), abs(d['bone_j'] - .5)
    out = {'transpose': off_i > off_j, 'conf_transpose': max(off_i, off_j) >= .05 and abs(off_i - off_j) >= .03}
    li, bj = (d['liver_j'], d['bone_i']) if out['transpose'] else (d['liver_i'], d['bone_j'])
    out['flip_i'], out['conf_flip_i'] = bool(li > .5), abs(li - .5) >= .05
    out['flip_j'] = bool(bj < .5) if np.isfinite(bj) else False
    out['conf_flip_j'] = bool(np.isfinite(bj) and abs(bj - .5) >= .05)
    out['flip_k'], out['conf_flip_k'] = bool(d['bulk_k'] > .5), abs(d['bulk_k'] - .5) >= .03
    return out

ORIENT = {f['patient_id']: decide(f) for f in FACTS if f['patient_id'] in INCLUDED}
orient_df = pd.DataFrame.from_dict(ORIENT, orient='index')
orientation_summary = {}
for part in ['transpose', 'flip_i', 'flip_j', 'flip_k']:
    confident = orient_df[orient_df[f'conf_{part}']]
    majority = bool(confident[part].mean() > .5) if len(confident) else False
    share = float((confident[part] == majority).mean()) if len(confident) else 0.0
    unclear = orient_df.index[~orient_df[f'conf_{part}']]
    orient_df.loc[unclear, part] = majority
    orientation_summary[part] = {'applied_to': int(orient_df[part].sum()), 'unclear_set_to_majority': int(len(unclear)),
                                 'majority': majority, 'agreement_among_clear': round(share, 3)}
    if len(unclear) > 0.10 * len(orient_df):                 # scans may differ from each other; too many unclear ones may not
        raise RuntimeError(f'STOP — orientation part {part!r} is unclear for {len(unclear)} scans (>10%). Send this output.')
ORIENT = orient_df[['transpose', 'flip_i', 'flip_j', 'flip_k']].astype(bool).to_dict('index')
orient_df.to_csv(OUT_DIR / 'E14b_mct_orientation_decisions.csv')
facts_df.to_csv(OUT_DIR / 'E14b_mct_facts.csv', index=False)

ok = facts_df[facts_df['technical_failure'] == '']
dicom = pd.DataFrame(dicom_rows)
FACTS_SUMMARY = {
    'in_plane_spacing_mm': [float(ok['spacing'].map(lambda s: s[0]).min()), float(ok['spacing'].map(lambda s: s[0]).median()),
                            float(ok['spacing'].map(lambda s: s[0]).max())],
    'slice_spacing_mm': [float(ok['spacing'].map(lambda s: s[2]).min()), float(ok['spacing'].map(lambda s: s[2]).median()),
                         float(ok['spacing'].map(lambda s: s[2]).max())],
    'tumour_components': ok['tumor_components'].value_counts().sort_index().to_dict(),
    'tumour_volume_ml_median': float(ok['tumor_volume_ml'].median()),
    'manufacturers': dicom['Manufacturer'].value_counts().to_dict() if 'Manufacturer' in dicom else {},
    'dicom_slice_thickness': dicom['SliceThickness'].value_counts().head(6).to_dict() if 'SliceThickness' in dicom else {},
    'kvp': dicom['KVP'].value_counts().head(6).to_dict() if 'KVP' in dicom else {},
}
print('Facts:', json.dumps(FACTS_SUMMARY, indent=1, default=str))
print('Orientation decisions (amendment 17.9-1):', json.dumps(orientation_summary, indent=1))

In [ ]:
# Cell 4 — E6 preprocessing, primary geometry: liver-only crops (gate) and crops at MCT-LTDiag's slice density (E14b)
INPLANE_MM = 400.0 / 512                       # internal physical pixel size (512 matrix, 400 mm field of view)
CROP_SIZE = 96
VARIANTS = ['primary', 'equalized']
CROP_DIR = WORK / 'crops'
CROP_DIR.mkdir(exist_ok=True)

def compute_center_of_mass(mask):             # Week 2, unchanged
    coords = np.array(np.where(mask > 0))
    if coords.shape[1] == 0:
        return None
    return tuple(int(round(c)) for c in coords.mean(axis=1))

def crop_cube(volume, center, size):          # Week 2, unchanged
    half = size // 2
    shape = volume.shape
    starts = [c - half for c in center]
    ends = [c + size - half for c in center]
    pad_before = [max(0, -s) for s in starts]
    pad_after = [max(0, e - shape[i]) for i, e in enumerate(ends)]
    s0, s1, s2 = (max(0, s) for s in starts)
    e0, e1, e2 = (min(shape[i], e) for i, e in enumerate(ends))
    cropped = volume[s0:e0, s1:e1, s2:e2]
    if any(pad_before) or any(pad_after):
        cropped = np.pad(cropped, [(pad_before[i], pad_after[i]) for i in range(3)], mode='constant', constant_values=0)
    return cropped

def resample(img, spacing, is_mask):
    size = [max(1, int(round(sz * sp / ts))) for sz, sp, ts in zip(img.GetSize(), img.GetSpacing(), spacing)]
    f = sitk.ResampleImageFilter()
    f.SetOutputSpacing(tuple(float(s) for s in spacing))
    f.SetSize(size)
    f.SetOutputOrigin(img.GetOrigin())
    f.SetOutputDirection(img.GetDirection())
    f.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    f.SetDefaultPixelValue(0 if is_mask else -1024)
    return f.Execute(img)

def apply_orientation(a, o):
    if o['transpose']:
        a = np.transpose(a, (1, 0, 2))
    if o['flip_i']:
        a = a[::-1]
    if o['flip_j']:
        a = a[:, ::-1]
    if o['flip_k']:
        a = a[:, :, ::-1]
    return np.ascontiguousarray(a)

def crops_from(arrays):
    liver, tumor = arrays['liver'] > 0, arrays['tumor'] > 0
    center = compute_center_of_mass(tumor) or compute_center_of_mass(liver)
    ct4 = np.stack([np.clip(crop_cube(arrays[p] * liver, center, CROP_SIZE), 0, 255).astype(np.uint8) for p in PHASE_ORDER])
    tumor_crop = (crop_cube(tumor.astype(np.uint8), center, CROP_SIZE) > 0).astype(np.uint8)
    meta = {'shape': list(tumor.shape), 'center': list(center), 'tumor_in_crop': float(tumor_crop.sum() / max(1, tumor.sum()))}
    return ct4, tumor_crop, meta

ANAT_E10 = pd.read_csv(INPUTS['e10b_anatomy'][0], dtype={'patient_id': str})
_geo = pd.read_csv(INPUTS['e12_mct_geometry'][0])
R_TARGET = float(_geo.loc[_geo['variant'] == 'equalized', 'ratio_before'].median())     # 00 log section 60: MCT-LTDiag's density
print('Target R (median liver slices / liver width, the 203 MCT-LTDiag patients, E12 file):', round(R_TARGET, 6))
if IN_COLAB and abs(R_TARGET - 0.1169811320754717) > 1e-12:
    raise RuntimeError('STOP — the target differs from the value fixed in the plan (0.116981). Send this output.')

def extent(mask, axis):                       # E10, unchanged
    idx = np.where(mask.any(axis=tuple(a for a in range(3) if a != axis)))[0]
    return (int(idx.min()), int(idx.max())) if idx.size else (-1, -1)

def density_ratio(liver):                     # E10b: slices containing liver / largest in-plane liver extent (pixels)
    lv = liver > 0
    n_liver = int(lv.any(axis=(0, 1)).sum())
    if n_liver == 0:
        return float('nan'), 0, 0
    li, lj = extent(lv, 0), extent(lv, 1)
    width = max(li[1] - li[0] + 1, lj[1] - lj[0] + 1)
    return n_liver / max(width, 1), n_liver, width

def resample_slices(vol, n_new, is_mask):     # E10, unchanged
    """Change only the number of slices (axis 2). CT linear; masks linear then > 0.5."""
    n_new = max(2, int(round(n_new)))
    if n_new == vol.shape[2]:
        return vol
    out = ndimage.zoom(vol.astype(np.float32), (1.0, 1.0, n_new / vol.shape[2]), order=1)
    assert out.shape[2] == n_new, (out.shape, n_new)
    return (out > 0.5).astype(np.float32) if is_mask else out

def slab_average(vol, n_new, is_mask):        # E14a (00 log section 60): thicker slices only, as a thick reconstruction
    """Fewer slices along axis 2: each new slice is the overlap-weighted mean of the original slices it covers. Masks are
    averaged the same way, then > 0.5. A scan is never made thinner: if n_new is not smaller, the volume is returned as it is."""
    n0 = vol.shape[2]
    n_new = max(2, int(round(n_new)))
    if n_new >= n0:
        return vol
    edges = np.arange(n_new + 1) * (n0 / n_new)
    weights = np.zeros((n0, n_new), dtype=np.float64)
    for k in range(n_new):
        for i in range(int(np.floor(edges[k])), min(n0, int(np.ceil(edges[k + 1])))):
            weights[i, k] = max(0.0, min(edges[k + 1], i + 1) - max(edges[k], i))
    weights /= weights.sum(axis=0, keepdims=True)
    out = np.tensordot(vol.astype(np.float32), weights.astype(np.float32), axes=([2], [0]))
    assert out.shape[2] == n_new, (out.shape, n_new)
    return (out > 0.5).astype(np.float32) if is_mask else out.astype(np.float32)

def adjust_slices(vol, n_new, is_mask):      # E14b (00 log section 60): thicker by slab averaging, thinner linearly (E12)
    n_new = max(2, int(round(n_new)))
    return slab_average(vol, n_new, is_mask) if n_new < vol.shape[2] else resample_slices(vol, n_new, is_mask)

def crops_equalized(arrays):                  # E14b: the equalization rule with MCT-LTDiag's target, then the primary crop rule
    ratio, n_liver, width = density_ratio(arrays['liver'])
    factor = R_TARGET / ratio if np.isfinite(ratio) else 1.0
    both = (arrays['liver'] > 0) | (arrays['tumor'] > 0)       # in-plane window: outside it the masked CT and both masks are 0
    ii, jj = np.where(both.any(axis=(1, 2)))[0], np.where(both.any(axis=(0, 2)))[0]
    win = (slice(int(ii.min()), int(ii.max()) + 1), slice(int(jj.min()), int(jj.max()) + 1)) if ii.size and jj.size else (slice(None), slice(None))
    n0 = arrays['liver'].shape[2]
    eq = {k: adjust_slices(np.ascontiguousarray(v[win]), n0 * factor, k in ('tumor', 'liver')) for k, v in arrays.items()}
    ct4, tumor_crop, meta = crops_from(eq)
    meta.update({'ratio_before': ratio, 'liver_slices': n_liver, 'liver_width_px': width, 'factor': float(factor),
                 'slices_before': int(n0), 'slices_after': int(eq['liver'].shape[2]), 'ratio_after': density_ratio(eq['liver'])[0]})
    return ct4, tumor_crop, meta

def preprocess(pid):
    target = CROP_DIR / f'{pid}.npz'
    meta_file = CROP_DIR / f'{pid}.json'
    if target.exists() and meta_file.exists():
        return json.loads(meta_file.read_text())
    folder = NII_DIR / pid
    facts = next(f for f in FACTS if f['patient_id'] == pid)
    imgs = {k: sitk.ReadImage(str(folder / f'{k}.nii.gz')) for k in PHASE_ORDER + ['tumor', 'liver']}
    for k in facts['off_grid']:                                  # rule 17.3-2
        imgs[k] = onto(imgs[k], imgs['C2'], is_mask=k in ('tumor', 'liver'))
    hu = facts['intensity_kind'] == 'HU'
    sz = imgs['C2'].GetSpacing()[2]
    settings = {'primary': (INPLANE_MM, INPLANE_MM, sz)}
    out, meta = {}, {}
    for variant, spacing in settings.items():
        vols = imgs if spacing is None else {k: resample(v, spacing, k in ('tumor', 'liver')) for k, v in imgs.items()}
        if variant == 'S5_week10g_axes':
            arrays = {k: sitk.GetArrayFromImage(v).astype(np.float32) for k, v in vols.items()}   # (k, j, i), as Week 10g
        else:
            arrays = {k: apply_orientation(ijk(v).astype(np.float32), ORIENT[pid]) for k, v in vols.items()}
        for p in PHASE_ORDER:
            arrays[p] = window_0255(arrays[p]) if hu else arrays[p]
        out[f'{variant}_ct'], out[f'{variant}_tumor'], meta[variant] = crops_from(arrays)
        out['equalized_ct'], out['equalized_tumor'], meta['equalized'] = crops_equalized(arrays)
    np.savez_compressed(target, **out)
    meta_file.write_text(json.dumps(meta))
    return meta

t0 = time.time()
CROP_META = {}
with ThreadPoolExecutor(N_IO) as pool:
    for n, (pid, meta) in enumerate(zip(INCLUDED, pool.map(preprocess, INCLUDED)), start=1):
        CROP_META[pid] = meta
        if n % 20 == 0 or n == len(INCLUDED):
            print(f'  preprocessed {n}/{len(INCLUDED)} ({(time.time() - t0) / 60:.1f} min)', flush=True)

crop_rows = [{'patient_id': pid, 'variant': v, **m} for pid, metas in CROP_META.items() for v, m in metas.items()]
pd.DataFrame(crop_rows).to_csv(OUT_DIR / 'E14b_mct_crop_geometry.csv', index=False)
(OUT_DIR / 'crops_equalized_mct').mkdir(exist_ok=True)
for pid in INCLUDED:
    with np.load(CROP_DIR / f'{pid}.npz') as z:
        np.savez_compressed(OUT_DIR / 'crops_equalized_mct' / f'{pid}.npz', ct=z['equalized_ct'], tumor_mask=z['equalized_tumor'])
geo = pd.DataFrame(crop_rows)
print('Tumour fraction inside the 96^3 crop (median by variant):', geo.groupby('variant')['tumor_in_crop'].median().round(3).to_dict())

In [ ]:
# Cell 5 — MCT-LTDiag crops in memory; gate 4: the liver-only (primary) crops must equal the saved E6 / E8.0 crops
E6_RUN = REVISION_DIR / 'E6_20261017T114646Z'
E8A_RUN = REVISION_DIR / 'E8a_20261017T162800Z'
if len(INCLUDED) != 203 and IN_COLAB:
    raise RuntimeError(f'STOP — {len(INCLUDED)} MCT-LTDiag patients included, expected 203.')
MCT_IDS = list(INCLUDED)
MCT_EQ, MCT_EQ_TM, diffs = {}, {}, []
for pid in MCT_IDS:
    with np.load(CROP_DIR / f'{pid}.npz') as z:
        MCT_EQ[pid], MCT_EQ_TM[pid], primary = z['equalized_ct'], z['equalized_tumor'], z['primary_ct']
    saved = E6_RUN / 'crops_primary' / f'{pid}.npz'
    saved = saved if saved.exists() else E8A_RUN / 'crops_primary_restored2' / f'{pid}.npz'
    with np.load(saved) as z:
        diffs.append(int(np.abs(z['ct'].astype(int) - primary.astype(int)).max()))
GATE = {'mct_primary_crops': {'patients_identical': int(sum(d == 0 for d in diffs)), 'max_abs_diff': int(max(diffs))}}
print('Gate 4 — MCT-LTDiag liver-only crops vs saved E6/E8.0 crops:', GATE['mct_primary_crops'])
if IN_COLAB and GATE['mct_primary_crops']['max_abs_diff'] > 1:
    raise RuntimeError('STOP — the MCT-LTDiag preprocessing does not reproduce the saved E6 crops. Send this output.')
MCT_GEO = pd.DataFrame({pid: CROP_META[pid]['equalized'] for pid in MCT_IDS}).T[
    ['ratio_before', 'factor', 'slices_before', 'slices_after', 'ratio_after']].astype(float)
print('MCT-LTDiag equalization (median): ratio before', round(float(MCT_GEO['ratio_before'].median()), 3), '| factor',
      round(float(MCT_GEO['factor'].median()), 2), '| slices before', float(MCT_GEO['slices_before'].median()), '| after',
      float(MCT_GEO['slices_after'].median()), '| ratio after', round(float(MCT_GEO['ratio_after'].median()), 3))
GATE['mct_median_ratio_equals_R'] = bool(abs(float(MCT_GEO['ratio_before'].median()) - R_TARGET) < 1e-9)
print('Gate 5 — recomputed MCT-LTDiag median ratio equals R:', GATE['mct_median_ratio_equals_R'])
if IN_COLAB and not GATE['mct_median_ratio_equals_R']:
    raise RuntimeError('STOP — the MCT-LTDiag ratio does not reproduce the E12 file. Send this output.')

In [ ]:
# Cell 6 — Internal data: labels (Week 5 table), folds, patient table with all four phases
ALL_IDS = dev_ids + test_ids
FOLD_OF = {str(p): f for f, s in enumerate(fold_splits) for p in s['val_ids']}
assert set(FOLD_OF) == set(dev_ids), 'fold files do not cover the development set exactly'
LABEL = internal.set_index('patient_id')['label'].astype(int).to_dict()
CLS = {p: CLASSES[LABEL[p]] for p in ALL_IDS}
table = patients_table[patients_table['patient_id'].isin(ALL_IDS)].drop_duplicates(['patient_id', 'phase'])   # Week 2: first row per phase
ROWS = {pid: g.set_index('phase') for pid, g in table.groupby('patient_id')}
missing = [p for p in ALL_IDS if p not in ROWS or any(ph not in ROWS[p].index for ph in PHASE_ORDER)]
if missing:
    raise RuntimeError(f'STOP — patients without all four phases: {missing[:10]}')
CROP, INJECT_RATIO, MIN_SHIFT, N_BOOT = 96, 0.29, 0.05, 2000      # E10's fixed values (00 log section 24)
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
N_CPU = os.cpu_count() or 2
print('Internal patients:', len(ALL_IDS), '| classes:', pd.Series(CLS).value_counts().to_dict())

In [ ]:
# Cell 7 — Internal crops for every condition in one pass over the scans (E10's Week 2 rebuild; E14b: MCT-LTDiag's density)
INT_DIR = WORK / 'internal'
INT_DIR.mkdir(exist_ok=True)
GATE3_IDS = list(ALL_IDS[:20])                # gate 3a: Week 5 features re-extracted from these patients' unequalized crops

def week2_crop(ct, tumor, liver):             # Week 2 process_patient, one phase: liver-only masking (as E11's liver_control)
    keep = liver > 0
    center = compute_center_of_mass(tumor) or compute_center_of_mass(liver)
    ct_crop = np.clip(crop_cube(ct * keep.astype(np.float32), center, CROP), 0, 255).astype(np.uint8)
    return ct_crop, crop_cube(tumor, center, CROP) > 0

def resampled(phase_vols, n_new):             # E14b: thicker by slab averaging, thinner linearly (adjust_slices, Cell 4)
    ct, tm, lv = phase_vols
    return [adjust_slices(ct, n_new, False), adjust_slices(tm, n_new, True), adjust_slices(lv, n_new, True)]

def process_patient(pid):
    rows = ROWS[pid]
    vols, anat = {}, {}
    for ph in PHASE_ORDER:
        ct = load_nii(DATA_DIR / rows.loc[ph, 'ct_path'])
        tumor, liver = load_nii(DATA_DIR / rows.loc[ph, 'mask_path']), load_nii(DATA_DIR / rows.loc[ph, 'liver_mask_path'])
        anat[ph] = density_ratio(liver) + (int(ct.shape[2]),)
        win = None                                                  # E10: in-plane window around liver and tumour (exact)
        if tumor.shape == ct.shape and liver.shape == ct.shape:
            both = (tumor > 0) | (liver > 0)
            ii, jj = np.where(both.any(axis=(1, 2)))[0], np.where(both.any(axis=(0, 2)))[0]
            if ii.size and jj.size:
                win = (slice(int(ii.min()), int(ii.max()) + 1), slice(int(jj.min()), int(jj.max()) + 1))
            del both
        vols[ph] = tuple(np.ascontiguousarray(a[win]) for a in (ct, tumor, liver)) if win else (ct, tumor, liver)
        del ct, tumor, liver
    pv_ratio = anat['C2'][0]
    def factor_for(liver):                                          # E12 rule, E14b target (00 log section 60)
        ratio = density_ratio(liver)[0]
        if np.isfinite(ratio):
            return R_TARGET / ratio, 'own'
        if np.isfinite(pv_ratio):
            return R_TARGET / pv_ratio, 'portal_venous'
        return 1.0, 'none'
    n_c2 = vols['C2'][0].shape[2]
    plan = {'liver_control': ({}, False), 'control': ({}, True)}     # condition -> (phase changes before equalization, equalize)
    if CLS[pid] in ('HCC', 'ICC'):
        plan['inject'] = ({'P': INJECT_RATIO * n_c2}, True)
    else:
        plan['repair'] = ({'P': n_c2}, True)
    stats = {'patient_id': pid, 'class': CLS[pid]}
    for ph in PHASE_ORDER:
        ratio, n_liver, width, n = anat[ph]
        stats.update({f'ratio_{ph}': ratio, f'liver_slices_{ph}': n_liver, f'liver_width_px_{ph}': width, f'n_before_{ph}': n})
    arrays = {}
    for name, (changes, equalize) in plan.items():                  # one condition at a time (memory)
        cts, tms, ns = [], [], []
        for ph in PHASE_ORDER:
            v = resampled(vols[ph], changes[ph]) if ph in changes else vols[ph]
            if equalize:
                factor, source = factor_for(v[2])
                v = resampled(v, v[0].shape[2] * factor)
                if name == 'control':
                    stats.update({f'factor_{ph}': float(factor), f'factor_source_{ph}': source, f'n_after_{ph}': int(v[0].shape[2]),
                                  f'ratio_after_{ph}': density_ratio(v[2])[0]})
            ct4, tm4 = week2_crop(*v)
            cts.append(ct4); tms.append(tm4); ns.append(v[0].shape[2])
            del v
        arrays[f'{name}_ct'], arrays[f'{name}_tm'], arrays[f'{name}_nslices'] = np.stack(cts), np.stack(tms), np.array(ns)
    np.savez_compressed(INT_DIR / f'{pid}.npz', **arrays)
    return pid, stats, [n for n in plan if n != 'liver_control']

def ram_text():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f' | RAM {vm.used / 1e9:.0f}/{vm.total / 1e9:.0f} GB'
    except Exception:
        return ''

def safe_process(pid):                                              # a data problem in one patient is recorded, not fatal
    try:
        return process_patient(pid) + (None,)
    except Exception as error:
        return pid, {}, [], f'{type(error).__name__}: {error}'

t0 = time.time()
STATS, CONDS, FAILED = [], {}, {}
N_PROC = max(1, min(6, N_CPU - 2)) if IN_COLAB else 1
def collect(results):
    for n, (pid, stats, conds, error) in enumerate(results, start=1):
        if error:
            FAILED[pid] = error; print(f'  {pid}: {error}', flush=True)
        else:
            STATS.append(stats); CONDS[pid] = conds
        if n % 25 == 0 or n == len(ALL_IDS):
            print(f'  patients processed {n}/{len(ALL_IDS)} ({(time.time() - t0) / 60:.1f} min){ram_text()}', flush=True)
done = False
if N_PROC > 1:
    try:
        collect(joblib.Parallel(n_jobs=N_PROC, return_as='generator')(joblib.delayed(safe_process)(p) for p in ALL_IDS))
        done = True
    except Exception as error:                                     # only infrastructure problems end up here
        print(f'  parallel processing failed ({error!r:.120}); continuing with threads')
        STATS.clear(); CONDS.clear(); FAILED.clear()
if not done:
    with ThreadPoolExecutor(3 if IN_COLAB else 2) as pool:
        collect(pool.map(safe_process, ALL_IDS))
if FAILED:
    pd.Series(FAILED).to_csv(OUT_DIR / 'E14b_failed_patients.csv')
    raise RuntimeError(f'STOP — {len(FAILED)} patients could not be processed (listed in E14b_failed_patients.csv): {dict(list(FAILED.items())[:5])}')
STATS = pd.DataFrame(STATS).set_index('patient_id').loc[ALL_IDS]
STATS.to_csv(OUT_DIR / 'E14b_internal_crop_stats.csv')

# ---- gate 1: unequalized liver-only crops equal the Week 2 processed crops; gate 2: liver slices and widths equal E10b ----
same = 0
for pid in ALL_IDS:
    with np.load(PROCESSED_DIR / f'{pid}.npz') as z, np.load(INT_DIR / f'{pid}.npz') as w:
        same += int(np.array_equal(z['ct'].astype(np.uint8), w['liver_control_ct']))
GATE['liver_only_crops_identical_to_week2'] = f'{same}/{len(ALL_IDS)}'
an = ANAT_E10.set_index(['patient_id', 'phase'])
mismatch = []
for pid in ALL_IDS:
    for ph in PHASE_ORDER:
        ref = an.loc[(pid, ph)]
        if int(ref['liver_slices']) != int(STATS.loc[pid, f'liver_slices_{ph}']) or \
                (int(ref['liver_slices']) > 0 and int(ref['liver_width_px']) != int(STATS.loc[pid, f'liver_width_px_{ph}'])):
            mismatch.append(f'{pid}/{ph}')
GATE['liver_slices_and_widths_equal_E10b'] = 'yes' if not mismatch else f'no: {mismatch[:10]}'
print('Gate 1 — unequalized liver-only crops identical to the Week 2 crops:', GATE['liver_only_crops_identical_to_week2'])
print('Gate 2 — liver slices and widths identical to E10b:', GATE['liver_slices_and_widths_equal_E10b'])
if IN_COLAB and (same < 0.99 * len(ALL_IDS) or mismatch):
    raise RuntimeError('STOP — gate 1 or 2 failed. Send this output.')
sources = pd.Series([STATS.loc[p, f'factor_source_{ph}'] for p in ALL_IDS for ph in PHASE_ORDER]).value_counts().to_dict()
print('Equalization factor sources (scans):', sources)
print('Median slices by class, portal venous, before -> after:',
      {c: (float(g['n_before_C2'].median()), float(g['n_after_C2'].median())) for c, g in STATS.groupby('class')})

In [ ]:
# Cell 8 — Train the phase-dropout CNN on the MCT-density crops (5 folds, E3 protocol: same folds, seeds, inner validation split)
import random
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
NUM_EPOCHS = int(os.environ.get('E14B_TEST_EPOCHS', 40))
BATCH_SIZE, LR, WD = 4, 1e-4, 1e-5
USE_AMP = DEVICE.type == 'cuda'
WORKERS = 4 if IN_COLAB else 0
CT = {}
for pid in ALL_IDS:
    with np.load(INT_DIR / f'{pid}.npz') as w:
        CT[pid] = w['control_ct']

# Cell 6 — Training code: E3 phase-dropout protocol, plus scanner-variation augmentation and the no-non-contrast option
def zscore4(x4):                              # Week 4 / E3, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def augment(vol4):                            # Week 4 / E3, unchanged
    if random.random() < 0.5:
        vol4 = vol4[:, ::-1].copy()
    if random.random() < 0.5:
        vol4 = vol4[:, :, ::-1].copy()
    if random.random() < 0.5:
        for c in range(vol4.shape[0]):
            scale = 1.0 + (random.random() - 0.5) * 0.2
            shift = (random.random() - 0.5) * 0.2
            mask = vol4[c] != 0
            vol4[c][mask] = vol4[c][mask] * scale + shift
    return vol4

def sample_kept(allowed):
    """E3 phase dropout over the allowed channels: all allowed with p = 0.5, else drop each with p = 0.5, keep >= 1."""
    if random.random() < 0.5:
        return tuple(allowed)
    while True:
        kept = tuple(c for c in allowed if random.random() >= 0.5)
        if kept:
            return kept

def keep_only(vol4, kept):
    out = vol4.copy()
    for c in range(4):
        if c not in kept:
            out[c] = 0.0
    return out

def domain_randomise(raw4):
    """Scanner-variation augmentation (plan 21, E8b-2) on the 0-255 crop before normalisation; one draw for all phases."""
    x = raw4.astype(np.float32)
    mask = x > 0
    if random.random() < 0.5:
        sigma = random.uniform(0.0, 1.5)
        x = np.stack([ndimage.gaussian_filter(x[c], sigma) for c in range(4)])
    if random.random() < 0.5:
        x = x + np.random.normal(0.0, random.uniform(0.0, 10.0), x.shape).astype(np.float32)
    if random.random() < 0.5:
        gamma = random.uniform(0.7, 1.5)
        x = 255.0 * (np.clip(x, 0, 255) / 255.0) ** gamma
    if random.random() < 0.5:
        f = random.uniform(0.5, 1.0)
        down = [ndimage.zoom(x[c], (f, f, 1.0), order=1) for c in range(4)]
        x = np.stack([ndimage.zoom(d, (96 / d.shape[0], 96 / d.shape[1], 1.0), order=1)[:96, :96, :96] for d in down])
    if random.random() < 0.5:
        sk = random.uniform(0.0, 2.0)
        x = np.stack([ndimage.gaussian_filter(x[c], (0, 0, sk)) for c in range(4)])
    if random.random() < 0.5:
        x = x * random.uniform(0.9, 1.1)
    return (np.clip(x, 0, 255) * mask).astype(np.float32)

class CropDataset(Dataset):
    def __init__(self, ids, arrays, train=False, phase_dropout=False, dr=False, allowed=(0, 1, 2, 3), kept=None):
        self.ids, self.arrays, self.train, self.phase_dropout, self.dr = list(ids), arrays, train, phase_dropout, dr
        self.allowed, self.kept = tuple(allowed), tuple(kept) if kept is not None else tuple(allowed)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, index):
        pid = self.ids[index]
        raw = self.arrays[pid].astype(np.float32)
        if self.train and self.dr:
            raw = domain_randomise(raw)
        vol4 = zscore4(raw)
        if self.train:
            vol4 = augment(vol4)
        kept = sample_kept(self.allowed) if (self.train and self.phase_dropout) else self.kept
        if kept != (0, 1, 2, 3):
            vol4 = keep_only(vol4, kept)
        return torch.from_numpy(np.ascontiguousarray(vol4, dtype=np.float32)), LABEL.get(pid, 0), pid

def seed_worker(worker_id):                  # each loader worker gets its own numpy stream (torch seeds python/torch per worker)
    np.random.seed(torch.initial_seed() % 2 ** 32)

def make_model():
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def class_weights_for(ids):
    counts = np.bincount([LABEL[p] for p in ids], minlength=3).astype(np.float32)
    return torch.from_numpy(counts.sum() / (3 * counts.clip(min=1))).float()

def macro_auc(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

@torch.no_grad()
def predict(model, ids, arrays, kept=(0, 1, 2, 3), batch_size=8):
    model.eval()
    loader = DataLoader(CropDataset(ids, arrays, kept=kept), batch_size=batch_size, shuffle=False,
                        num_workers=2 if IN_COLAB else 0, pin_memory=USE_AMP)
    probs, pids = [], []
    for x, _, batch_ids in loader:
        probs.append(torch.softmax(model(x.to(DEVICE, non_blocking=True)).float().cpu(), dim=1).numpy())
        pids.extend(batch_ids)
    return pd.DataFrame(np.concatenate(probs), index=pids, columns=P3)

def set_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if USE_AMP:
        torch.cuda.manual_seed_all(seed)

def train_fold(arm, cfg, fold):
    path = OUT_DIR / arm / f'fold_{fold}.pt'
    if path.exists():
        print(f'  {arm} fold {fold}: already trained, loading.')
        return
    path.parent.mkdir(exist_ok=True)
    outer_train = list(fold_splits[fold]['train_ids'])
    inner_train, inner_val = train_test_split(outer_train, test_size=0.2, random_state=SEED + fold,
                                              stratify=[LABEL[p] for p in outer_train])
    set_seeds(SEED + fold)
    loader = DataLoader(CropDataset(inner_train, CT, train=True, phase_dropout=True, dr=cfg['dr'], allowed=cfg['allowed']),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS, pin_memory=USE_AMP, drop_last=True,
                        persistent_workers=WORKERS > 0, worker_init_fn=seed_worker)
    model = make_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for(inner_train).to(DEVICE))
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    y_inner = np.array([LABEL[p] for p in inner_val])
    best_auc, best_state, best_epoch, history = -1.0, None, -1, []
    for epoch in range(NUM_EPOCHS):
        model.train()
        started, loss_sum, n = time.time(), 0.0, 0
        for x, y, _ in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * x.size(0)
            n += x.size(0)
        scheduler.step()
        inner_auc = macro_auc(y_inner, predict(model, inner_val, CT, kept=cfg['allowed']).loc[inner_val].to_numpy())
        history.append({'epoch': epoch, 'train_loss': loss_sum / max(n, 1), 'inner_val_auc': inner_auc, 'seconds': time.time() - started})
        if inner_auc > best_auc:
            best_auc, best_epoch = inner_auc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save({'state_dict': best_state, 'arm': arm, 'fold': fold, 'best_epoch': best_epoch, 'best_inner_val_auc': best_auc,
                'config': {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}, 'epochs': NUM_EPOCHS}, path)
    pd.DataFrame(history).to_csv(OUT_DIR / arm / f'fold_{fold}_history.csv', index=False)
    print(f'  {arm} fold {fold}: best inner-val AUC {best_auc:.4f} at epoch {best_epoch} ({sum(h["seconds"] for h in history) / 60:.1f} min)', flush=True)
    del model
    if USE_AMP:
        torch.cuda.empty_cache()

def load_state(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = make_model().to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model


ARM, ARM_CFG = 'mct_density_phase_dropout', {'dr': False, 'allowed': (0, 1, 2, 3)}
t0 = time.time()
for fold in range(5):
    train_fold(ARM, ARM_CFG, fold)
print(f'Training done in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Cell 9 — Radiomics: Week 5 extraction on the MCT-density crops (internal conditions and MCT-LTDiag); E2 recipe refitted; gate 3
import lightgbm as lgb
from sklearn.feature_selection import f_classif
VERSIONS['lightgbm'] = lgb.__version__
RADIOMICS_PARAMS = {                          # Week 5, unchanged (as E10)
    'setting': {'binWidth': 5, 'label': 1, 'force2D': False, 'normalize': False, 'resampledPixelSpacing': None},
    'imageType': {'Original': {}},
    'featureClass': {'firstorder': None, 'shape': None, 'glcm': None, 'glrlm': None, 'glszm': None, 'gldm': None, 'ngtdm': None},
}
def extract(ct4, tm):                         # tm: per-phase masks (4, 96, 96, 96) or one mask for all phases (MCT-LTDiag)
    extractor = featureextractor.RadiomicsFeatureExtractor(RADIOMICS_PARAMS)
    feats = {}
    for i, phase in enumerate(PHASE_ORDER):
        mask = tm[i] if tm.ndim == 4 else tm
        try:
            result = extractor.execute(sitk.GetImageFromArray(ct4[i].astype(np.float32)), sitk.GetImageFromArray(mask.astype(np.uint8)))
        except Exception:
            continue
        for key, value in result.items():
            if not key.startswith('diagnostics_'):
                try:
                    feats[f'{phase}_{key}'] = float(value)
                except (TypeError, ValueError):
                    pass
    return feats
def internal_job(pid, conds):
    conds = list(conds) + (['liver_control'] if pid in GATE3_IDS else [])
    with np.load(INT_DIR / f'{pid}.npz') as w:
        return pid, {cond: extract(w[f'{cond}_ct'], w[f'{cond}_tm']) for cond in conds}
def mct_job(pid):
    with np.load(CROP_DIR / f'{pid}.npz') as z:
        return pid, extract(z['equalized_ct'], z['equalized_tumor'])
def run_jobs(function, jobs):
    if N_JOBS > 1:
        try:
            return joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(function)(*job) for job in jobs)
        except Exception as error:            # same results, only slower
            print(f'  parallel radiomics failed ({error!r:.120}); continuing one patient at a time')
    return [function(*job) for job in jobs]

t0 = time.time()
feat_int, feat_mct = OUT_DIR / 'E14b_radiomics_features_internal.csv.gz', OUT_DIR / 'E14b_radiomics_features_mct.csv.gz'
if feat_int.exists():                         # resume: reuse the features extracted before
    RADF = pd.read_csv(feat_int, dtype={'patient_id': str})
else:
    res = run_jobs(internal_job, [(pid, CONDS[pid]) for pid in ALL_IDS])
    RADF = pd.DataFrame([{'patient_id': pid, 'condition': cond, **f} for pid, d in res for cond, f in d.items()])
    RADF.to_csv(feat_int, index=False, compression='gzip')
if feat_mct.exists():
    RADM = pd.read_csv(feat_mct, dtype={'patient_id': str}).set_index('patient_id')
else:
    RADM = pd.DataFrame.from_dict(dict(run_jobs(mct_job, [(pid,) for pid in MCT_IDS])), orient='index')
    RADM.index.name = 'patient_id'
    RADM.to_csv(feat_mct, compression='gzip')
print(f'Radiomics: internal {RADF.shape}, MCT-LTDiag {RADM.shape} ({(time.time() - t0) / 60:.1f} min)')

POOL = [c for c in internal.columns if c not in ('patient_id', 'label') and '_original_shape_' not in c]   # E2: 372 features
K_TOP_FEATURES, CORR_THRESHOLD = 30, 0.95
def select_features_train(X_tr, y_tr, feature_names, k=K_TOP_FEATURES, corr_thresh=CORR_THRESHOLD):   # Week 5 / E2, unchanged
    stds = np.nanstd(X_tr, axis=0)
    keep = stds > 1e-10
    X_kept = X_tr[:, keep]
    kept_idx_global = np.where(keep)[0]
    X_nan_to_num = np.nan_to_num(X_kept, nan=np.nanmedian(X_kept))
    corr = np.abs(np.corrcoef(X_nan_to_num.T))
    remove = np.zeros(corr.shape[0], dtype=bool)
    for i in range(corr.shape[0]):
        if remove[i]:
            continue
        for j in range(i + 1, corr.shape[0]):
            if remove[j]:
                continue
            if corr[i, j] > corr_thresh:
                remove[j] = True
    kept_idx_after_corr = kept_idx_global[~remove]
    X_for_f = SimpleImputer(strategy='median').fit_transform(X_tr[:, kept_idx_after_corr])
    F, _ = f_classif(X_for_f, y_tr)
    order = np.argsort(F)[::-1]
    final_idx = kept_idx_after_corr[order[:min(k, len(order))]]
    return final_idx, [feature_names[i] for i in final_idx]

def fit_radiomics(df, cols):                  # E7a's E2 recipe, keeping the fold models (for conditions and MCT-LTDiag)
    oof, test, models = [], [], []
    test_X = df.set_index('patient_id').loc[test_ids, cols].to_numpy(dtype=np.float32)
    for s in fold_splits:
        tr = df[df['patient_id'].isin(s['train_ids'])]
        va = df.set_index('patient_id').loc[s['val_ids']]
        X_tr, y_tr = tr[cols].to_numpy(dtype=np.float32), tr['label'].to_numpy()
        idx, _ = select_features_train(X_tr, y_tr, cols)
        imp = SimpleImputer(strategy='median').fit(X_tr[:, idx])
        sc = StandardScaler().fit(imp.transform(X_tr[:, idx]))
        clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, num_leaves=15, min_child_samples=5,
                                 reg_alpha=0.1, reg_lambda=0.1, class_weight='balanced', random_state=42, verbose=-1)
        clf.fit(sc.transform(imp.transform(X_tr[:, idx])), y_tr)
        oof.append(pd.DataFrame(clf.predict_proba(sc.transform(imp.transform(va[cols].to_numpy(dtype=np.float32)[:, idx]))), index=s['val_ids'], columns=P3))
        test.append(clf.predict_proba(sc.transform(imp.transform(test_X[:, idx]))))
        models.append({'feature_pool': cols, 'sel_idx': idx, 'imputer': imp, 'scaler': sc, 'clf': clf})
    return pd.concat(oof).loc[dev_ids], pd.DataFrame(np.mean(test, axis=0), index=test_ids, columns=P3), models

# gate 3a: features re-extracted from 20 unequalized crops equal Week 5; gate 3b: the E2 recipe reproduces the saved E2 predictions
ctrl = RADF[RADF['condition'] == 'control'].set_index('patient_id')
lc = RADF[RADF['condition'] == 'liver_control'].set_index('patient_id')
ref_feat = internal.set_index('patient_id').loc[GATE3_IDS, POOL].astype(float)
rel = (lc.loc[GATE3_IDS].reindex(columns=POOL).astype(float) - ref_feat).abs() / (ref_feat.abs() + 1e-6)
GATE['radiomics_unchanged_patients_vs_week5'] = float(np.nanmax(rel.to_numpy()))
e2_oof_check, _, _ = fit_radiomics(internal, POOL)
e2_saved = pd.read_csv(INPUTS['e2_radiomics_oof'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids, P3]
GATE['radiomics_recipe_vs_E2'] = float(np.abs(e2_oof_check.to_numpy() - e2_saved.to_numpy()).max())
print('Gate 3 — 20 unequalized crops vs Week 5 features (max relative difference):', GATE['radiomics_unchanged_patients_vs_week5'],
      '| E2 recipe vs saved E2 out-of-fold predictions (max difference):', GATE['radiomics_recipe_vs_E2'])
if IN_COLAB and (GATE['radiomics_unchanged_patients_vs_week5'] > 1e-6 or GATE['radiomics_recipe_vs_E2'] > 1e-6):
    raise RuntimeError('STOP — gate 3 failed (radiomics extraction or recipe). Send this output.')

eq_df = ctrl.loc[ALL_IDS].reindex(columns=POOL).reset_index()
eq_df['label'] = eq_df['patient_id'].map(LABEL)
RAD_OOF, RAD_TEST, RAD_MODELS = fit_radiomics(eq_df, POOL)
def rad_probs(models, X):
    return np.mean([m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X[:, m['sel_idx']].astype(np.float32)))) for m in models], axis=0)
rows = []
for _, r in RADF.iterrows():                  # development patients: own fold model; held-out: the 5-model ensemble (as E10)
    pid = r['patient_id']
    X = r.reindex(POOL).to_numpy(dtype=np.float64)[None, :]
    models = [RAD_MODELS[FOLD_OF[pid]]] if pid in FOLD_OF else RAD_MODELS
    rows.append({'model': 'radiomics', 'patient_id': pid, 'condition': r['condition'], **dict(zip(P3, rad_probs(models, X)[0]))})
PRED = pd.DataFrame(rows)
XM = RADM.reindex(index=MCT_IDS, columns=POOL).to_numpy(dtype=np.float64)
GATE['mct_radiomics_missing_cells'] = int((~np.isfinite(XM)).sum())
MCT_NEW = {'radiomics': pd.DataFrame(rad_probs(RAD_MODELS, XM), index=MCT_IDS, columns=P3)}
check = PRED[(PRED['condition'] == 'control')].set_index('patient_id')
assert np.allclose(check.loc[dev_ids, P3].to_numpy(), RAD_OOF.loc[dev_ids].to_numpy())
print('Radiomics refitted on the new features; MCT-LTDiag cells missing:', GATE['mct_radiomics_missing_cells'])

In [ ]:
# Cell 10 — MCT-density CNN on every condition and on MCT-LTDiag; meta-learners refitted on the new out-of-fold predictions
U_MODELS = [load_state(OUT_DIR / ARM / f'fold_{f}.pt') for f in range(5)]
@torch.no_grad()
def cnn_predict(pid, ct4):                    # E10: own fold model for development patients, ensemble for held-out
    x = torch.from_numpy(zscore4(ct4.astype(np.float32))).unsqueeze(0).to(DEVICE)
    models = [U_MODELS[FOLD_OF[pid]]] if pid in FOLD_OF else U_MODELS
    return np.mean([torch.softmax(m(x).float(), dim=1).cpu().numpy()[0] for m in models], axis=0)
rows = []
for pid in ALL_IDS:
    with np.load(INT_DIR / f'{pid}.npz') as w:
        for cond in CONDS[pid]:
            rows.append({'model': 'CNN', 'patient_id': pid, 'condition': cond, **dict(zip(P3, cnn_predict(pid, w[f'{cond}_ct'])))})
PRED = pd.concat([PRED, pd.DataFrame(rows)], ignore_index=True)
with torch.no_grad():
    MCT_NEW['CNN'] = pd.DataFrame([np.mean([torch.softmax(m(torch.from_numpy(zscore4(MCT_EQ[p].astype(np.float32))).unsqueeze(0).to(DEVICE)).float(), dim=1).cpu().numpy()[0]
                                            for m in U_MODELS], axis=0) for p in MCT_IDS], index=MCT_IDS, columns=P3)
ctl = PRED[PRED['condition'] == 'control'].set_index(['model', 'patient_id'])[P3]
CNN_OOF, CNN_TEST = ctl.loc['CNN'].loc[dev_ids], ctl.loc['CNN'].loc[test_ids]

W4, W5 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC'], ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
FUSIONS = {'CNN + radiomics': W4 + W5, 'full fusion': W4 + W5 + ['age', 'gender']}
oof_meta = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids].reset_index()
test_meta = pd.read_csv(INPUTS['e2_test_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids]
oof_meta[W4], oof_meta[W5] = CNN_OOF.loc[oof_meta['patient_id']].to_numpy(), RAD_OOF.loc[oof_meta['patient_id']].to_numpy()
def prepare(X):                               # E8b, unchanged
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X
def fit_meta_folds(columns):                  # E8b meta_folds, unchanged
    fitted = []
    for f in range(5):
        tr = oof_meta['fold'].astype(int).ne(f).to_numpy()
        X = prepare(oof_meta.loc[tr, columns].to_numpy())
        imp = SimpleImputer(strategy='median').fit(X)
        sc = StandardScaler().fit(imp.transform(X))
        m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), oof_meta.loc[tr, 'label'].astype(int))
        fitted.append((imp, sc, m))
    return fitted
def apply_meta(fitted, X):
    return np.mean([m.predict_proba(s.transform(i.transform(prepare(X)))) for i, s, m in fitted], axis=0)
META = {name: fit_meta_folds(cols) for name, cols in FUSIONS.items()}
clinical = pd.concat([oof_meta.set_index('patient_id')[['age', 'gender']], test_meta[['age', 'gender']]])
cnn_c = PRED[PRED['model'] == 'CNN'].set_index(['patient_id', 'condition'])[P3]
rad_c = PRED[PRED['model'] == 'radiomics'].set_index(['patient_id', 'condition'])[P3]
rows = []
for (pid, cond) in cnn_c.index:
    if (pid, cond) not in rad_c.index:
        continue
    for name, cols in FUSIONS.items():
        if name == 'full fusion' and cond != 'control':
            continue                              # the artefact test uses CNN + radiomics (as E10)
        x = np.concatenate([cnn_c.loc[(pid, cond)].to_numpy(), rad_c.loc[(pid, cond)].to_numpy()]
                           + ([clinical.loc[pid, ['age', 'gender']].to_numpy(dtype=np.float64)] if name == 'full fusion' else []))[None, :]
        fitted = [META[name][FOLD_OF[pid]]] if pid in FOLD_OF else META[name]
        rows.append({'model': name, 'patient_id': pid, 'condition': cond, **dict(zip(P3, apply_meta(fitted, x)[0]))})
PRED = pd.concat([PRED, pd.DataFrame(rows)], ignore_index=True)
PRED.to_csv(OUT_DIR / 'E14b_predictions_internal.csv', index=False)

def parse_age(value):
    match = re.fullmatch(r'0*(\d+)\s*[Yy]?', str(value).strip())
    return float(match.group(1)) if match else np.nan
clin_m = SEL.set_index('patient_id').loc[MCT_IDS]
AGE, SEX = clin_m['patient_age'].map(parse_age).to_numpy(), clin_m['patient_sex'].str.strip().str.upper().map({'M': 1.0, 'F': 0.0}).to_numpy()
for name, cols in FUSIONS.items():
    frame = np.column_stack([MCT_NEW['CNN'].to_numpy(), MCT_NEW['radiomics'].to_numpy()] + ([AGE, SEX] if name == 'full fusion' else []))
    MCT_NEW[name] = pd.DataFrame(apply_meta(META[name], frame), index=MCT_IDS, columns=P3)
pd.concat([f.assign(model=n).rename_axis('patient_id').reset_index() for n, f in MCT_NEW.items()]).to_csv(OUT_DIR / 'E14b_predictions_mct.csv', index=False)
print('Predictions: internal', PRED.shape, '| MCT-LTDiag models', list(MCT_NEW))

In [ ]:
# Cell 11 — Results O0-O3, pre-specified rules (00 log section 60), saved outputs
from scipy.stats import norm
MAIN_KEYS = {'CNN': ('CNN phase_dropout_CNN [complete]', 'CNN phase_dropout_CNN'),
             'radiomics': ('radiomics radiomics_noshape', 'radiomics radiomics_noshape'),
             'CNN + radiomics': ('cnn_plus_radiomics: [ref] phase_dropout_CNN + radiomics_noshape',) * 2,
             'full fusion': ('full_fusion: [ref] phase_dropout_CNN + radiomics_noshape',) * 2}
e8b = pd.read_csv(INPUTS['e8b_predictions'][0], dtype={'patient_id': str})
def main_pred(model, cohort, ids):
    key = MAIN_KEYS[model][0 if cohort.startswith('internal') else 1]
    t = e8b[(e8b['model'] == key) & (e8b['cohort'] == cohort)].set_index('patient_id')
    return t.loc[ids, P3]
newp = PRED[PRED['condition'] == 'control'].set_index(['model', 'patient_id'])[P3]
INT = {}
for m in MAIN_KEYS:
    INT[('E14b', m)] = (newp.loc[m].loc[dev_ids], newp.loc[m].loc[test_ids])
    INT[('main', m)] = (main_pred(m, 'internal_oof', dev_ids), main_pred(m, 'internal_test', test_ids))
y_dev, y_test = np.array([LABEL[p] for p in dev_ids]), np.array([LABEL[p] for p in test_ids])
rng = np.random.default_rng(SEED)
BOOT_T = [b for b in (rng.integers(0, len(test_ids), len(test_ids)) for _ in range(N_BOOT)) if len(np.unique(y_test[b])) == 3]

# ---- O0: equalization check ----
eq_rows, spreads = [], {}
for ph in PHASE_ORDER:
    for quantity in ['ratio', 'ratio_after', 'n_before', 'n_after']:
        med = STATS.groupby('class')[f'{quantity}_{ph}'].median()
        eq_rows.append({'phase': ph, 'quantity': quantity, **{c: float(med.get(c, np.nan)) for c in CLASSES},
                        'max_over_min': float(med.max() / med.min()) if med.min() > 0 else np.nan})
    spreads[ph] = eq_rows[-3]['max_over_min']                       # ratio_after
EQ_TABLE = pd.DataFrame(eq_rows)
EQ_OK = all(np.isfinite(s) and s <= 1.25 for s in spreads.values())
def slice_count_model(prefix):                  # multinomial logistic regression on the four log slice counts
    X = np.log(STATS[[f'{prefix}_{ph}' for ph in PHASE_ORDER]].astype(float).clip(lower=1))
    pos = {p: i for i, p in enumerate(dev_ids)}
    def fit(ids):
        sc = StandardScaler().fit(X.loc[ids].to_numpy())
        m = LogisticRegression(max_iter=2000, random_state=SEED).fit(sc.transform(X.loc[ids].to_numpy()), [LABEL[p] for p in ids])
        return lambda q: m.predict_proba(sc.transform(X.loc[q].to_numpy()))
    held = macro_auc(y_test, fit(dev_ids)(test_ids))
    oof = np.zeros((len(dev_ids), 3))
    for s in fold_splits:
        oof[[pos[str(p)] for p in s['val_ids']]] = fit([str(p) for p in s['train_ids']])([str(p) for p in s['val_ids']])
    two = np.isin(y_dev, [0, 1])
    return {'slices': prefix, 'heldout_macro_auc': held, 'oof_macro_auc': macro_auc(y_dev, oof),
            'oof_hcc_vs_icc_auc': float(roc_auc_score(y_dev[two] == 0, oof[two, 0] / np.clip(oof[two, 0] + oof[two, 1], 1e-12, None)))}
O0 = pd.DataFrame([slice_count_model('n_before'), slice_count_model('n_after')])

# ---- O1: internal ----
o1, paired = [], []
for (version, m), (oof, test) in INT.items():
    p = test.to_numpy(); pred = p.argmax(1)
    vals = [macro_auc(y_test[b], p[b]) for b in BOOT_T]
    o1.append({'pipeline': version, 'model': m, 'heldout_macro_auc': macro_auc(y_test, p), 'boot_low': float(np.percentile(vals, 2.5)),
               'boot_high': float(np.percentile(vals, 97.5)), 'oof_macro_auc': macro_auc(y_dev, oof.to_numpy()),
               'heldout_accuracy': float((pred == y_test).mean()),
               **{f'recall_{c}': float(((pred == i) & (y_test == i)).sum() / (y_test == i).sum()) for i, c in enumerate(CLASSES)}})
for m in MAIN_KEYS:
    a, b = INT[('E14b', m)][1].to_numpy(), INT[('main', m)][1].to_numpy()
    d = np.array([macro_auc(y_test[i], a[i]) - macro_auc(y_test[i], b[i]) for i in BOOT_T])
    paired.append({'model': m, 'heldout_difference_E14b_minus_main': macro_auc(y_test, a) - macro_auc(y_test, b),
                   'boot_low': float(np.percentile(d, 2.5)), 'boot_high': float(np.percentile(d, 97.5)),
                   'bootstrap_p': float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))),
                   'oof_difference_E14b_minus_main': macro_auc(y_dev, INT[('E14b', m)][0].to_numpy()) - macro_auc(y_dev, INT[('main', m)][0].to_numpy())})
O1, O1_PAIRED = pd.DataFrame(o1), pd.DataFrame(paired)

# ---- O2: residual artifact (injection or repair before equalization, against the unaltered equalized scan) ----
def boot_ci(v):
    v = np.asarray(v, dtype=float)
    means = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(N_BOOT)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))
TESTS = [('non-contrast artifact injected before equalization (HCC, ICC)', 'inject', [p for p in ALL_IDS if CLS[p] in ('HCC', 'ICC')], +1),
         ('non-contrast artifact repaired before equalization (cHCC-CCA)', 'repair', [p for p in ALL_IDS if CLS[p] == 'CHCC'], -1)]
PI = PRED.set_index(['model', 'patient_id', 'condition'])[P3]
eff = []
for model in ['CNN', 'radiomics', 'CNN + radiomics']:
    for name, cond, ids, sign in TESTS:
        for subset, sub_ids in [('all 278', ids), ('held-out 56', [p for p in ids if p in set(test_ids)])]:
            ok_ids = [p for p in sub_ids if all((model, p, c) in PI.index for c in (cond, 'control'))]
            if not ok_ids:
                continue
            d = np.array([PI.loc[(model, p, cond), 'p_CHCC'] - PI.loc[(model, p, 'control'), 'p_CHCC'] for p in ok_ids])
            lo, hi = boot_ci(d)
            eff.append({'model': model, 'test': name, 'subset': subset, 'n': len(ok_ids), 'expected_sign': sign,
                        'mean_shift_vs_equalized_scan': float(d.mean()), 'ci_low': lo, 'ci_high': hi,
                        'meets_E10_thresholds': bool(sign * d.mean() >= MIN_SHIFT and (lo > 0 if sign > 0 else hi < 0))})
O2 = pd.DataFrame(eff)
def meets(model, cond):
    e = O2[(O2['model'] == model) & (O2['subset'] == 'all 278') & O2['test'].str.contains(cond)]
    return bool(len(e) and e['meets_E10_thresholds'].all())

# ---- O3: MCT-LTDiag, HCC versus ICC (E11 code) ----
def compute_midrank(values):                  # DeLong (E4d/E6, unchanged)
    order = np.argsort(values); sorted_values = values[order]; n_values = len(values)
    midranks = np.zeros(n_values, dtype=np.float64); index = 0
    while index < n_values:
        end = index
        while end < n_values and sorted_values[end] == sorted_values[index]:
            end += 1
        midranks[index:end] = 0.5 * (index + end - 1) + 1; index = end
    output = np.empty(n_values, dtype=np.float64); output[order] = midranks
    return output
def delong_components(y, s):
    pos, neg = s[y == 1], s[y == 0]; m, n = len(pos), len(neg)
    tx, ty, tz = compute_midrank(pos), compute_midrank(neg), compute_midrank(np.concatenate([pos, neg]))
    return tz[:m].sum() / m / n - (m + 1.0) / 2.0 / n, (tz[:m] - tx) / n, 1.0 - (tz[m:] - ty) / m
def delong_ci(y, s):
    auc, v01, v10 = delong_components(y, s)
    se = np.sqrt(np.var(v01, ddof=1) / len(v01) + np.var(v10, ddof=1) / len(v10))
    return float(auc), float(max(0, auc - 1.96 * se)), float(min(1, auc + 1.96 * se))
def delong_paired_p(y, s1, s2):
    a1, v01a, v10a = delong_components(y, s1); a2, v01b, v10b = delong_components(y, s2)
    cov = np.cov(np.vstack([v01a, v01b])) / len(v01a) + np.cov(np.vstack([v10a, v10b])) / len(v10a)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    return 1.0 if var <= 0 else float(2 * norm.sf(abs(a1 - a2) / np.sqrt(var)))
Y_MCT = (SEL.set_index('patient_id').loc[MCT_IDS, 'type'] == 'HCC').astype(int).to_numpy()
def score(frame):
    p = frame.loc[MCT_IDS].to_numpy()
    return p[:, 0] / np.clip(p[:, 0] + p[:, 1], 1e-12, None), p.argmax(1)
o3, o3p = [], []
for m in MAIN_KEYS:
    for version, frame in [('E14b', MCT_NEW[m]), ('main', main_pred(m, 'MCT_LTDiag', MCT_IDS))]:
        s, pred = score(frame)
        auc, lo, hi = delong_ci(Y_MCT, s)
        o3.append({'pipeline': version, 'model': m, 'auc': auc, 'delong_low': lo, 'delong_high': hi,
                   'hcc_sensitivity': float(((pred == 0) & (Y_MCT == 1)).sum() / Y_MCT.sum()),
                   'icc_recall': float(((pred == 1) & (Y_MCT == 0)).sum() / (1 - Y_MCT).sum()),
                   'specificity_icc_not_called_hcc': float(((pred != 0) & (Y_MCT == 0)).sum() / (1 - Y_MCT).sum()),
                   'HCC_pred_HCC/ICC/CHCC': '/'.join(str(int(((pred == k) & (Y_MCT == 1)).sum())) for k in range(3)),
                   'ICC_pred_HCC/ICC/CHCC': '/'.join(str(int(((pred == k) & (Y_MCT == 0)).sum())) for k in range(3))})
    s_new, _ = score(MCT_NEW[m]); s_main, _ = score(main_pred(m, 'MCT_LTDiag', MCT_IDS))
    o3p.append({'model': m, 'auc_difference_E14b_minus_main': float(roc_auc_score(Y_MCT, s_new) - roc_auc_score(Y_MCT, s_main)),
                'delong_p': delong_paired_p(Y_MCT, s_new, s_main)})
O3, O3_PAIRED = pd.DataFrame(o3), pd.DataFrame(o3p)

# ---- the pre-specified statements (00 log section 60) ----
cr = O1_PAIRED.set_index('model').loc['CNN + radiomics']
ext = O3_PAIRED.set_index('model').loc['CNN + radiomics']
ext_new = O3[(O3['pipeline'] == 'E14b') & (O3['model'] == 'CNN + radiomics')].iloc[0]
INTERNAL_CHANGED, INTERNAL_FELL = bool(cr['boot_low'] > 0 or cr['boot_high'] < 0), bool(cr['boot_high'] < 0)
EXTERNAL_IMPROVED = bool(ext['delong_p'] < 0.05 and ext['auc_difference_E14b_minus_main'] > 0)
INTERPRETATION = ([] if INTERNAL_CHANGED or EXTERNAL_IMPROVED else ['acquisition geometry is not the main cause of the external failure'])
INTERPRETATION += ['acquisition mismatch contributed to the external failure'] if EXTERNAL_IMPROVED else []
INTERPRETATION += ['internal discrimination depended on class-dependent acquisition'] if INTERNAL_FELL else []
INTERPRETATION += ['internal performance rose; no pre-specified reading'] if INTERNAL_CHANGED and not INTERNAL_FELL and not EXTERNAL_IMPROVED else []
RULES = {
    'harmonization': ('achieved: class medians of the ratio within 1.25-fold in every phase' if EQ_OK else
                     'not achieved in ' + ', '.join(f'{ph} ({s:.2f})' for ph, s in spreads.items() if not (np.isfinite(s) and s <= 1.25))),
    'slice_count_only_heldout_macro_auc_before_after': [round(float(v), 3) for v in O0['heldout_macro_auc']],
    'internal': {m: ('changed (CI excludes 0)' if (r['boot_low'] > 0 or r['boot_high'] < 0) else 'not changed')
                 for m, r in O1_PAIRED.set_index('model').iterrows()},
    'internal_CNN_plus_radiomics': (f"changed by {cr['heldout_difference_E14b_minus_main']:+.3f} ({cr['boot_low']:+.3f} to {cr['boot_high']:+.3f})"
                                    if (cr['boot_low'] > 0 or cr['boot_high'] < 0) else 'matching the slice density to MCT-LTDiag did not measurably change internal performance'),
    'external_CNN_plus_radiomics': ('matching the slice density to MCT-LTDiag improved external discrimination' if EXTERNAL_IMPROVED
                                    else 'matching the slice density to MCT-LTDiag did not improve external discrimination'),
    'external_CNN_plus_radiomics_vs_chance': (f"AUC {ext_new['auc']:.3f} (DeLong 95% CI {ext_new['delong_low']:.3f} to {ext_new['delong_high']:.3f}): "
                                              + ('better than chance' if ext_new['delong_low'] > 0.5 else 'not significantly better than chance')),
    'interpretation': INTERPRETATION,
    'residual_artifact_CNN': ('the MCT-density CNN still responds to the thick reconstruction' if meets('CNN', 'injected')
                              else 'injection did not meet the E10 thresholds for the MCT-density CNN'),
    'residual_artifact_CNN_repair_meets_thresholds': meets('CNN', 'repaired'),
}

for name, frame in [('E14b_equalization_by_class', EQ_TABLE), ('E14b_slice_count_only_model', O0), ('E14b_internal_results', O1),
                    ('E14b_internal_paired_differences', O1_PAIRED), ('E14b_residual_artifact_effects', O2),
                    ('E14b_mct_results', O3), ('E14b_mct_paired', O3_PAIRED)]:
    frame.to_csv(OUT_DIR / f'{name}.csv', index=False)
summary = {'run_id': RUN_ID, 'notebook': 'R18_E14b_mct_density_retraining.ipynb', 'versions': VERSIONS, 'input_sha256': input_hashes, 'gates': GATE,
           'epochs': NUM_EPOCHS, 'R_target': R_TARGET, 'rules': RULES,
           'protocol': '00 log section 60, E14b (written 23 Oct 2026, before the run); the main models are unchanged'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
            for p in sorted(OUT_DIR.rglob('*')) if p.is_file() and p.suffix != '.npz']
pd.DataFrame(manifest).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E14b_{RUN_ID} | epochs {NUM_EPOCHS} | R = {R_TARGET:.4f} | gates: {GATE}')
print('\n--- O0 equalization by class (medians) ---'); print(EQ_TABLE.round(3).to_string(index=False))
print('\n--- O0 slice-count-only model, before and after equalization ---'); print(O0.round(3).to_string(index=False))
print('\n--- O1 internal (held-out 56; out-of-fold 222) ---'); print(O1.round(3).to_string(index=False))
print('\n--- O1 paired differences, E14b minus main (held-out) ---'); print(O1_PAIRED.round(3).to_string(index=False))
print('\n--- O2 residual artifact, equalized models ---'); print(O2.round(3).to_string(index=False))
print('\n--- O3 MCT-LTDiag (203), HCC vs ICC ---'); print(O3.round(3).to_string(index=False))
print('\n--- O3 paired, E14b minus main ---'); print(O3_PAIRED.round(3).to_string(index=False))
print('\nPre-specified statements:', json.dumps(RULES, indent=1, default=str))
print('MCT-LTDiag slice-density factors (median):', round(float(MCT_GEO['factor'].median()), 3), '| internal factors by class, portal venous:',
      {c: round(float(g['factor_C2'].median()), 3) for c, g in STATS.groupby('class')})
print('==============================================================')

In [ ]:
# Cell 12 — Finish: make sure every result file has reached Google Drive, then disconnect the runtime (stops compute units)
SHUTDOWN_WHEN_DONE = True     # set to False to keep the runtime after the run
if IN_COLAB and SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('All files are saved in Drive. The runtime disconnects in 30 seconds; the outputs above stay visible on this page.')
    time.sleep(30)
    runtime.unassign()